# 03 - Assign Cluster Labels to ALL Trajectories  (Stage 3)

**Purpose.** Load the chosen k-means model **and the saved StandardScaler**, and
assign a `cluster_label` to every trajectory by nearest-centroid lookup
(`predict`, no iteration). The scaler from notebook 02 must be applied to the raw
features before `predict`, so every trajectory is standardized with the *same*
means/stds the model was trained on.

- `complete` and `partial` trajectories are labelled (for `partial`, the last
  known position was already used as the proxy for the missing later day(s) in
  Stage 1).
- `early_loss` trajectories have no usable features -> `cluster_label = -1`.

If `config.GROUP_MAP` is set, a merged `cluster_group` column is also added;
otherwise `cluster_group` mirrors `cluster_label`.

**Input.** `data/features.parquet`, `data/kmeans_models/kmeans_k{BEST_K}.pkl`,
`data/kmeans_models/scaler.pkl`.
**Output.** `data/labeled_trajectories.parquet`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # project root: config.py, pipeline.py
import numpy as np
import pandas as pd
import config as C
import pipeline as P
print("project root:", C.PROJECT_ROOT)
print("sampling days:", C.DAYS, "| feature space:", C.FEATURE_SPACE)

project root: /work/bk1450/b383184/Amazon/Mercator/notebooks/Analysis/kmeans_analysis/kmean_analysis_30_180_stdZ_w
sampling days: [30, 50, 100, 150, 180] | feature space: zscore


## 3.1  Choose k and load the model + scaler

In [2]:
BEST_K = 50          # <-- set after inspecting notebook 02 (papermill: -p BEST_K <k>)

In [3]:
import pickle
with open(C.MODELS_DIR / f"kmeans_k{BEST_K}.pkl", "rb") as f:
    km = pickle.load(f)
with open(C.MODELS_DIR / "scaler.pkl", "rb") as f:
    scaler = pickle.load(f)
print("loaded model with", km.n_clusters, "clusters and the shared StandardScaler")

loaded model with 50 clusters and the shared StandardScaler


## 3.2  Predict labels for every labellable trajectory

In [4]:
features = pd.read_parquet(C.FEATURES_FILE)
labelable = features.status != "early_loss"
# Standardize with the SAME scaler, then apply the SAME per-day weighting the
# model was fit with (config.DAY_WEIGHTS via pipeline.feature_weight_vector()).
# The k-means centroids live in this weighted space, so predict MUST see it too;
# omitting * W would assign labels in a different space than the model was fit in.
W = P.feature_weight_vector()
X_all = scaler.transform(P.build_feature_matrix(features[labelable])) * W
labels = np.full(len(features), -1, dtype=np.int32)
labels[labelable.to_numpy()] = km.predict(X_all).astype(np.int32)
features["cluster_label"] = labels
print(features.cluster_label.value_counts().sort_index().to_string())

cluster_label
-1      120000
 0      771831
 1      191899
 2     1087453
 3      302451
 4      232059
 5      167121
 6       75828
 7      145733
 8      501423
 9      228481
 10     105814
 11     129817
 12     146240
 13     212759
 14     630452
 15     138950
 16     226986
 17     152851
 18     145007
 19     165437
 20    1154599
 21     139605
 22     104387
 23     238173
 24     113718
 25     887645
 26     264494
 27     143998
 28     264066
 29      98878
 30     154125
 31      95607
 32     885845
 33     602638
 34     106145
 35    2175036
 36     137169
 37      96591
 38     271860
 39     176163
 40      73387
 41     115808
 42     621374
 43     176894
 44     427742
 45     169533
 46     215971
 47     112865
 48      94817
 49     282275


## 3.2b  Merge into pathway groups (optional)

If `config.GROUP_MAP` is non-empty, merge the raw clusters into groups; the
result is stored in `cluster_group`. With an empty map, `cluster_group` simply
equals `cluster_label`.

In [5]:
if C.GROUP_MAP:
    groups, raw2grp = P.apply_group_map(features.cluster_label.to_numpy(), C.GROUP_MAP, BEST_K)
    features["cluster_group"] = groups.astype(np.int32)
    print("raw cluster -> group:", raw2grp)
    print(features.loc[features.cluster_group >= 0, "cluster_group"]
          .value_counts().sort_index().to_string())
else:
    features["cluster_group"] = features["cluster_label"]
    print("GROUP_MAP empty -> cluster_group mirrors cluster_label")

raw cluster -> group: {0: 5, 1: 0, 2: 5, 3: 1, 4: 1, 5: 1, 6: 6, 7: 0, 8: 4, 9: 3, 10: 2, 11: 1, 12: 0, 13: 0, 14: 4, 15: 0, 16: 3, 17: 2, 18: 2, 19: 1, 20: 5, 21: 0, 22: 2, 23: 1, 24: 1, 25: 5, 26: 4, 27: 3, 28: 0, 29: 3, 30: 1, 31: 3, 32: 5, 33: 4, 34: 2, 35: 5, 36: 0, 37: 0, 38: 1, 39: 1, 40: 6, 41: 2, 42: 5, 43: 0, 44: 5, 45: 0, 46: 0, 47: 2, 48: 0, 49: 1}
cluster_group
0    2130227
1    2233199
2     842877
3     793950
4    1999007
5    8011525
6     149215


## 3.3  Save labelled trajectories

In [6]:
features.to_parquet(C.LABELED_FILE, index=False)
print("saved", features.shape, "->", C.LABELED_FILE)

saved (16280000, 18) -> /work/bk1450/b383184/Amazon/Mercator/notebooks/Analysis/kmeans_analysis/kmean_analysis_30_180_stdZ_w/data/labeled_trajectories.parquet


## 3.4  Summary

In [7]:
lbl = features[features.cluster_label >= 0]
print(f"Labelled {len(lbl):,} trajectories into {BEST_K} clusters "
      f"({(features.cluster_label==-1).sum():,} early_loss left unlabelled).")
print(f"Saved to {C.LABELED_FILE}")

Labelled 16,160,000 trajectories into 50 clusters (120,000 early_loss left unlabelled).
Saved to /work/bk1450/b383184/Amazon/Mercator/notebooks/Analysis/kmeans_analysis/kmean_analysis_30_180_stdZ_w/data/labeled_trajectories.parquet
